In [3]:
!pip install sentence-transformers


   ---------------------------------------- 0.0/571.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/571.3 kB ? eta -:--:--
   ---------------------------------------- 571.3/571.3 kB 2.5 MB/s  0:00:00



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
import os

# 1. Tải dữ liệu gốc
print("Đang tải dữ liệu 20 Newsgroups...")
data = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
X_text = np.array(data.data)
Y_labels = np.array(data.target)

def create_exponential_longtail(X, Y, imbalance_ratio=0.1):
    """
    Tạo dữ liệu lệch theo đường cong mũ: n_i = N_max * (ratio ^ (i / (C-1)))
    """
    unique_classes = np.unique(Y)
    num_classes = len(unique_classes)
    
    X_lt, Y_lt = [], []
    
    # In ra tiêu đề để theo dõi phân phối
    print(f"{'Lớp':<10} | {'Số lượng gốc':<15} | {'Số lượng giữ lại (Mũ)':<20}")
    print("-" * 50)

    for cls in unique_classes:
        # Lấy index của lớp hiện tại
        idx = np.where(Y == cls)[0]
        np.random.shuffle(idx)
        
        # SỐ LƯỢNG GỐC của lớp này
        n_max = len(idx)
        
        # CÔNG THỨC ĐƯỜNG CONG MŨ:
        # Lớp 0 (cls=0) sẽ giữ lại: n_max * (ratio^0) = n_max * 1 (Giữ hết)
        # Lớp 19 (cls=19) sẽ giữ lại: n_max * (ratio^1) = n_max * ratio (Giữ 10% nếu ratio=0.1)
        exponent = cls / (num_classes - 1)
        num_to_keep = int(n_max * (imbalance_ratio ** exponent))
        
        # In ra để kiểm tra độ dốc
        print(f"Lớp {cls:<6} | {n_max:<15} | {num_to_keep:<20}")
        
        X_lt.extend(X[idx[:num_to_keep]])
        Y_lt.extend([cls] * num_to_keep)
        
    return X_lt, np.array(Y_lt)

# Thực thi tạo Long-tail
# imbalance_ratio=0.1 nghĩa là lớp cuối cùng chỉ bằng 10% lớp đầu tiên
X_lt_text, Y_lt_text = create_exponential_longtail(X_text, Y_labels, imbalance_ratio=0.1)

print(f"\nTổng số văn bản sau khi làm Long-tail đường cong mũ: {len(X_lt_text)}")

Đang tải dữ liệu 20 Newsgroups...
Lớp        | Số lượng gốc    | Số lượng giữ lại (Mũ)
--------------------------------------------------
Lớp 0      | 799             | 799                 


MemoryError: Unable to allocate 484. MiB for an array with shape (799,) and data type <U158791

In [4]:
from sentence_transformers import SentenceTransformer

print("Đang trích xuất đặc trưng văn bản (Sentence-BERT)...")
# all-MiniLM-L6-v2 cho kết quả 384 chiều, rất nhanh và mạnh
model = SentenceTransformer('all-MiniLM-L6-v2') 

X_features_text = model.encode(X_lt_text, batch_size=64, show_progress_bar=True)

# Lưu lại để chạy Grid Testing
os.makedirs('../data/text_processed', exist_ok=True)
np.save('../data/text_processed/X_features_text_exp.npy', X_features_text)
np.save('../data/text_processed/Y_longtail_text_exp.npy', Y_lt_text)

print("Xong! Dữ liệu đã sẵn sàng để Grid Testing.")

c:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đang trích xuất đặc trưng văn bản (Sentence-BERT)...


c:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ACER\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6746.52it/s]
BertMode

NameError: name 'X_lt_text' is not defined

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import entropy

# Cấu hình 10 cases thử nghiệm
test_configs = [
    {"T": 2, "m": 5,  "k_list": [600, 300]},
    {"T": 2, "m": 10, "k_list": [600, 300]},
    {"T": 2, "m": 20, "k_list": [600, 300]}, # Test m lớn
    {"T": 3, "m": 5,  "k_list": [800, 500, 300]},
    {"T": 3, "m": 10, "k_list": [800, 500, 300]},
    {"T": 3, "m": 20, "k_list": [800, 500, 300]},
    {"T": 3, "m": 10, "k_list": [1000, 600, 400]}, # Test k_final lớn hơn
    {"T": 4, "m": 5,  "k_list": [1200, 800, 500, 300]},
    {"T": 4, "m": 10, "k_list": [1200, 800, 500, 300]},
    {"T": 4, "m": 15, "k_list": [1200, 800, 500, 300]}, # Test T cực sâu
]

results = []

for config in test_configs:
    T, m, k_list = config["T"], config["m"], config["k_list"]
    k_final = k_list[-1]
    
    print(f"Testing Case: T={T}, m={m}, k_final={k_final}...")
    
    t0 = time.time()
    
    # 1. Chạy HK-means (Hàm của bạn)
    centroids = hierarchical_kmeans_resampling(X_text_norm, k_list, T, m, num_init=1)
    
    # 2. Gán nhãn
    km = KMeans(n_clusters=k_final, init=centroids, n_init=1).fit(X_text_norm)
    labels = km.labels_
    
    # 3. Nhặt ảnh (Logic avg // 2 của bạn)
    avg_per_cluster = len(X_text_norm) // k_final
    target = max(1, avg_per_cluster // 2)
    
    curated_idx = []
    for i in range(k_final):
        cluster_idx = np.where(labels == i)[0]
        if len(cluster_idx) == 0: continue
        d = np.linalg.norm(X_text_norm[cluster_idx] - centroids[i], axis=1)
        curated_idx.extend(cluster_idx[np.argsort(d)[:min(len(cluster_idx), target)]])
    
    t1 = time.time()
    
    # 4. Đo đạc
    res_entropy = entropy(np.unique(Y_text_lt[curated_idx], return_counts=True)[1] / len(curated_idx))
    
    results.append({
        "Config": f"T{T}m{m}k{k_final}",
        "Time": t1 - t0,
        "Entropy": res_entropy,
        "N": len(curated_idx)
    })

df = pd.DataFrame(results)

In [ ]:
fig, ax1 = plt.subplots(figsize=(15, 7))

# Đường Entropy
ax1.plot(df['Config'], df['Entropy'], color='tab:blue', marker='o', linewidth=3, label='Entropy')
ax1.set_ylabel('Entropy (Càng cao càng phẳng)', color='tab:blue', fontsize=12)
ax1.tick_params(axis='y', labelcolor='tab:blue')

# Đường Thời gian
ax2 = ax1.twinx()
ax2.plot(df['Config'], df['Time'], color='tab:red', marker='s', linestyle='--', linewidth=3, label='Thời gian')
ax2.set_ylabel('Thời gian thực thi (Giây)', color='tab:red', fontsize=12)
ax2.tick_params(axis='y', labelcolor='tab:red')

# Chú thích số lượng mẫu N ngay trên biểu đồ
for i in range(len(df)):
    ax1.annotate(f"N={df['N'][i]}", (df['Config'][i], df['Entropy'][i]), 
                 xytext=(0, 10), textcoords='offset points', ha='center', fontweight='bold')

plt.title('TEXT GRID TESTING: TÁC ĐỘNG CỦA T, m VÀ k ĐẾN HIỆU QUẢ LÀM PHẲNG', fontsize=14)
ax1.set_xticklabels(df['Config'], rotation=45)
ax1.grid(axis='both', linestyle=':', alpha=0.6)
fig.tight_layout()
plt.show()

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import entropy

# Cấu hình 10 cases thử nghiệm
test_configs = [
    {"T": 2, "m": 5,  "k_list": [600, 300]},
    {"T": 2, "m": 10, "k_list": [600, 300]},
    {"T": 2, "m": 20, "k_list": [600, 300]}, # Test m lớn
    {"T": 3, "m": 5,  "k_list": [800, 500, 300]},
    {"T": 3, "m": 10, "k_list": [800, 500, 300]},
    {"T": 3, "m": 20, "k_list": [800, 500, 300]},
    {"T": 3, "m": 10, "k_list": [1000, 600, 400]}, # Test k_final lớn hơn
    {"T": 4, "m": 5,  "k_list": [1200, 800, 500, 300]},
    {"T": 4, "m": 10, "k_list": [1200, 800, 500, 300]},
    {"T": 4, "m": 15, "k_list": [1200, 800, 500, 300]}, # Test T cực sâu
]

results = []

for config in test_configs:
    T, m, k_list = config["T"], config["m"], config["k_list"]
    k_final = k_list[-1]
    
    print(f"Testing Case: T={T}, m={m}, k_final={k_final}...")
    
    t0 = time.time()
    
    # 1. Chạy HK-means (Hàm của bạn)
    centroids = hierarchical_kmeans_resampling(X_text_norm, k_list, T, m, num_init=5)
    
    # 2. Gán nhãn
    km = KMeans(n_clusters=k_final, init=centroids, n_init=5).fit(X_text_norm)
    labels = km.labels_
    
    # 3. Nhặt ảnh (Logic avg // 2 của bạn)
    avg_per_cluster = len(X_text_norm) // k_final
    target = max(1, avg_per_cluster // 2)
    
    curated_idx = []
    for i in range(k_final):
        cluster_idx = np.where(labels == i)[0]
        if len(cluster_idx) == 0: continue
        d = np.linalg.norm(X_text_norm[cluster_idx] - centroids[i], axis=1)
        curated_idx.extend(cluster_idx[np.argsort(d)[:min(len(cluster_idx), target)]])
    
    t1 = time.time()
    
    # 4. Đo đạc
    res_entropy = entropy(np.unique(Y_text_lt[curated_idx], return_counts=True)[1] / len(curated_idx))
    
    results.append({
        "Config": f"T{T}m{m}k{k_final}",
        "Time": t1 - t0,
        "Entropy": res_entropy,
        "N": len(curated_idx)
    })

df = pd.DataFrame(results)

In [ ]:
fig, ax1 = plt.subplots(figsize=(15, 7))

# Đường Entropy
ax1.plot(df['Config'], df['Entropy'], color='tab:blue', marker='o', linewidth=3, label='Entropy')
ax1.set_ylabel('Entropy (Càng cao càng phẳng)', color='tab:blue', fontsize=12)
ax1.tick_params(axis='y', labelcolor='tab:blue')

# Đường Thời gian
ax2 = ax1.twinx()
ax2.plot(df['Config'], df['Time'], color='tab:red', marker='s', linestyle='--', linewidth=3, label='Thời gian')
ax2.set_ylabel('Thời gian thực thi (Giây)', color='tab:red', fontsize=12)
ax2.tick_params(axis='y', labelcolor='tab:red')

# Chú thích số lượng mẫu N ngay trên biểu đồ
for i in range(len(df)):
    ax1.annotate(f"N={df['N'][i]}", (df['Config'][i], df['Entropy'][i]), 
                 xytext=(0, 10), textcoords='offset points', ha='center', fontweight='bold')

plt.title('TEXT GRID TESTING: TÁC ĐỘNG CỦA T, m VÀ k ĐẾN HIỆU QUẢ LÀM PHẲNG', fontsize=14)
ax1.set_xticklabels(df['Config'], rotation=45)
ax1.grid(axis='both', linestyle=':', alpha=0.6)
fig.tight_layout()
plt.show()